# 04 YOLO Smoke Test On MacBook

Purpose: run a tiny YOLO detection smoke test before any full training. This notebook is intended to confirm the local pipeline, not to produce a deployable model.

**Inputs**

- Prepared detection dataset, or access to the cleaned dataset so the script can prepare a small subset
- Ultralytics installed in the active environment
- Optional local YOLO weights or explicit permission to download weights

**Outputs**

- `outputs/experiments/`
- `outputs/run_all_summary.json`


In [ ]:
from pathlib import Path
import os
import sys

def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists() and (p / "scripts").exists() and (p / "src").exists():
            return p
    raise RuntimeError("Could not find repo root. Open the notebook from inside the fishDetect repository.")

REPO = find_repo_root()
os.chdir(REPO)

DATASET_ROOT = Path(os.environ.get(
    "FISHDETECT_DATASET_ROOT",
    "/Users/ec/Documents/Data/FishDetectNOAA/_data/merged_viame_v2"
))

PREPARED_ROOT = Path(os.environ.get(
    "FISHDETECT_PREPARED_ROOT",
    "/Users/ec/Documents/Data/FishDetectNOAA/_data/prepared_ml"
))

os.environ["FISHDETECT_DATASET_ROOT"] = str(DATASET_ROOT)
os.environ["FISHDETECT_PREPARED_ROOT"] = str(PREPARED_ROOT)

if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

print("Repo root:", REPO)
print("Dataset root:", DATASET_ROOT)
print("Prepared root:", PREPARED_ROOT)
print("Dataset exists:", DATASET_ROOT.exists())
print("Prepared parent exists:", PREPARED_ROOT.parent.exists())


## Device and Dependency Check


In [ ]:
import importlib.util

try:
    import torch
    print("Torch:", torch.__version__)
    print("MPS available:", bool(getattr(torch.backends, "mps", None) and torch.backends.mps.is_available()))
    print("CUDA available:", torch.cuda.is_available())
except Exception as exc:
    print("Torch check failed:", exc)

if importlib.util.find_spec("ultralytics") is None:
    print("Ultralytics is not installed. Install intentionally with:")
    print("python -m pip install ultralytics")
else:
    import ultralytics
    print("Ultralytics:", getattr(ultralytics, "__version__", "installed"))


## Optional Tiny Preparation

If the prepared dataset does not exist yet, run a small preparation step first. It writes at most 50 images to the prepared root and leaves the master dataset unchanged.


In [ ]:
RUN_TINY_PREPARE_IF_NEEDED = True

needed = [PREPARED_ROOT / "yolo_det" / "data.yaml", PREPARED_ROOT / "metadata" / "split_manifest.csv"]
if RUN_TINY_PREPARE_IF_NEEDED and not all(path.exists() for path in needed):
    !python scripts/prepare_dataset.py --config configs/experiments.yaml --max-images 50
else:
    print("Prepared dataset is already present, or tiny preparation is disabled.")


## Run Smoke Group Only

This command runs the conservative smoke group. If weights are unavailable and downloads are disabled, the experiment is skipped and the reason is recorded instead of failing cryptically.


In [ ]:
!python scripts/run_all_experiments.py --group smoke --smoke-test --max-images 50 --epochs 1


## Smoke Summary


In [ ]:
import json
from pathlib import Path
from IPython.display import display

summary_path = Path("outputs/run_all_summary.json")
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    display(summary)
    for row in summary.get("experiments", []):
        print(f"{row.get('name')}: {row.get('status')} {row.get('reason', '')}")
else:
    print("No smoke summary exists yet.")


This notebook does not launch full training. To allow an intentional Ultralytics weight download, use the command-line workflow with `--allow-downloads`.
